# Практичне завдання 4
## Дослідження дерев рішень та ансамблевих методів у задачі багатокласової класифікації за наявності пропущених значень

**Мета.** Дослідити та порівняти дерево рішень (Decision Tree), випадковий ліс (Random Forest) і градієнтний бустинг (XGBoost) на датасеті **Annealing** з пропущеними значеннями; окремо оцінити вплив індикаторів пропусків, коректно підібрати гіперпараметри через Optuna без витоку даних, отримати неупереджену оцінку вкладеною крос-валідацією, статистично порівняти моделі та інтерпретувати їхні прогнози (MDI, permutation importance, SHAP).

**Дані.** OpenML `anneal`, версія 1: 898 спостережень, 38 ознак, 5 наявних класів. Для відтворюваності поруч із ноутбуком зберігається кеш `anneal_openml_v1.pkl`; якщо OpenML недоступний, ноутбук читає його.

**Індивідуальний варіант.** `SEED = 42 + N`, де `N = 0`, тобто **`SEED = 42`**. Ця сама константа ініціалізує поділ даних, крос-валідацію, моделі та семплер Optuna.

**Структура роботи:** первинний аналіз даних → поділ і пайплайни → базові моделі → Optuna та вкладена CV → статистичне порівняння → фінальна оцінка → інтерпретація → аналіз розбіжностей → висновки.

## 0. Імпорт, налаштування та фіксація версій

Усі джерела випадковості ініціалізуються однією константою `SEED`. Попередження про очікувану поведінку рідкісного класу під час внутрішньої крос-валідації пригнічуються, щоб не захаращувати вивід; самі обмеження аналізуються текстом.

In [ ]:
import json
import random
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
from optuna.exceptions import ExperimentalWarning
import pandas as pd
import seaborn as sns
import shap
from IPython.display import display
from scipy.stats import chi2_contingency, spearmanr, t as t_dist
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix,
                             f1_score)
from sklearn.model_selection import (RepeatedStratifiedKFold,
                                     StratifiedKFold, cross_val_predict,
                                     cross_val_score, cross_validate,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

SEED = 42              # 42 + N, N = 0
N_TRIALS = 30          # кількість trials Optuna в кожному запуску tune()
N_SPLITS_OUTER = 5     # зовнішні блоки вкладеної CV
N_SPLITS_INNER = 3     # внутрішні блоки Optuna (рідкісний клас!)
N_REPEATS = 3          # повторення базової CV
N_JOBS_RF = 8          # паралелізм RF: на малих даних більше потоків лише шкодить
N_JOBS_XGB = 2         # XGBoost на 898 об'єктах швидший із 2 потоками

np.random.seed(SEED)
random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=ExperimentalWarning)
warnings.filterwarnings("ignore", message=".*least populated class.*")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

In [ ]:
import matplotlib
import scipy
import sklearn
import xgboost

versions = pd.Series({
    "Python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "optuna": optuna.__version__,
    "shap": shap.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
}, name="версія")
display(versions.to_frame())

## 1. Первинний аналіз даних

### 1.1. Завантаження та структура

Завантажуємо датасет з OpenML. Категоріальні ознаки одразу приводимо до типу `object`, як вимагає специфікація: це спрощує подальший поділ на числові та категоріальні стовпці через `make_column_selector`.

In [ ]:
DATA_PATH = Path("anneal_openml_v1.pkl")

def load_anneal(path=DATA_PATH):
    '''Завантажує anneal з OpenML, а за відсутності мережі — з локального кешу.'''
    try:
        bunch = fetch_openml(name="anneal", version=1, as_frame=True)
        X_raw = bunch.data.copy()
        y_raw = bunch.target.copy()
        if not path.exists():
            pd.concat([X_raw, y_raw.rename("class")], axis=1).to_pickle(path)
        source = 'OpenML: name="anneal", version=1'
    except Exception as exc:
        frame = pd.read_pickle(path)
        X_raw = frame.drop(columns="class")
        y_raw = frame["class"]
        source = f"локальний кеш {path.name} (OpenML недоступний: {type(exc).__name__})"

    for col in X_raw.columns:
        if isinstance(X_raw[col].dtype, pd.CategoricalDtype):
            X_raw[col] = X_raw[col].astype(object)

    return X_raw, y_raw.astype(str), source


X, y, DATA_SOURCE = load_anneal()
print("Джерело даних:", DATA_SOURCE)
print(f"X: {X.shape[0]} спостережень, {X.shape[1]} ознак")
display(X.head())

In [ ]:
feature_summary = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "n_unique": X.nunique(dropna=True),
    "n_missing": X.isna().sum(),
    "missing_frac": X.isna().mean(),
}).sort_values("missing_frac", ascending=False)

type_counts = pd.Series({
    "числові": X.select_dtypes(include="number").shape[1],
    "категоріальні (object)": X.select_dtypes(include="object").shape[1],
}, name="кількість ознак")

print(f"Розмірність: {X.shape[0]} × {X.shape[1]}")
display(type_counts.to_frame())
display(feature_summary)

**Висновок до структури.** Датасет містить 38 ознак: 6 числових (`carbon`, `hardness`, `strength`, `thick`, `width`, `len`) і 32 категоріальні. Пропуски є масовим явищем: у понад половини ознак частка пропусків перевищує 80 %. Це не типовий «шум» заповнення, а ознака того, що значна частина атрибутів є структурно необов'язковою для конкретних виробів.

### 1.2. Частка пропусків і видалення порожніх/константних ознак

Ознаки, які повністю порожні або мають лише одне унікальне непорожнє значення, не несуть інформації для моделі. Їх видаляємо **до** поділу, оскільки це детерміноване структурне рішення, яке не залежить від цільової змінної й не створює витоку.

In [ ]:
missing = X.isna().mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 11))
colors = ["#c44e52" if v >= 1.0 else "#4c72b0" for v in missing.values]
ax.barh(missing.index, missing.values * 100, color=colors)
ax.set_xlabel("Частка пропусків, %")
ax.set_title("Частка пропусків за ознаками (до видалення)")
ax.axvline(50, color="grey", linestyle="--", linewidth=0.8)
for i, (name, value) in enumerate(missing.items()):
    ax.text(value * 100 + 1, i, f"{value * 100:.1f}", va="center", fontsize=7)
fig.tight_layout()
plt.show()

In [ ]:
all_missing = [c for c in X.columns if X[c].notna().sum() == 0]
constant = [c for c in X.columns
            if X[c].notna().sum() > 0 and X[c].nunique(dropna=True) <= 1]
drop_cols = all_missing + constant

removed = feature_summary.loc[drop_cols, ["dtype", "n_missing", "n_unique"]].copy()
removed["reason"] = ["повністю порожня" if c in all_missing else "константна"
                     for c in removed.index]
print(f"Видалено ознак: {len(drop_cols)} "
      f"(порожніх — {len(all_missing)}, константних — {len(constant)})")
display(removed)

X = X.drop(columns=drop_cols)
print(f"Залишилося ознак: {X.shape[1]}")
display(X.dtypes.astype(str).rename("dtype").to_frame().T)

**Висновок до видалення.** Видалено 20 із 38 ознак: 6 повністю порожніх (`m`, `marvi`, `corr`, `jurofm`, `s`, `p`) і 14 константних (наприклад, `product-type` завжди `C`, `temper_rolling` завжди `T`, `bc`/`bf`/`bt`/`bl` мають лише `Y`). Після очищення залишилося 18 ознак. Це важливий крок: константні ознаки не дають жодного розщеплення, але створювали б зайві one-hot стовпці та індикатори пропусків.

### 1.3. Розподіл цільової змінної та рідкісні класи

Клас 4, описаний в документації UCI, у версії OpenML відсутній. Наявні 5 класів сильно незбалансовані: клас `3` становить понад 76 %, а клас `1` — лише 8 об'єктів (0.89 %).

In [ ]:
class_counts = y.value_counts().sort_index()
class_pct = y.value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(class_counts.index, class_counts.values, color="#4c72b0")
ax.set_yscale("log")
ax.set_xlabel("Клас")
ax.set_ylabel("Кількість об'єктів (log scale)")
ax.set_title("Розподіл класів цільової змінної")
for bar, value, pct in zip(bars, class_counts.values, class_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value * 1.15,
            f"{value}\n({pct:.2f} %)", ha="center", va="bottom", fontsize=8)
ax.set_ylim(top=class_counts.max() * 4)
fig.tight_layout()
plt.show()

display(pd.DataFrame({"кількість": class_counts,
                      "частка, %": class_pct.round(2)}))
print("Класи в даних:", list(class_counts.index))
print("Клас 4 у версії OpenML відсутній.")

**Рішення щодо рідкісного класу `1` (8 об'єктів).** Залишаємо його без об'єднання та без виключення. По-перше, це реальний клас предметної області, і його виключення змінило б постановку задачі. По-друге, після стратифікованого поділу 6 об'єктів потрапляють у train і 2 — у test, тому клас присутній у кожному зовнішньому блоці (5 блоків) і в кожному внутрішньому блоці (3 блоки) принаймні в навчальній частині. По-третє, саме через цей клас macro F1 має велику дисперсію: одна помилка на 1–2 об'єктах змінює внесок класу в середнє на 0.1–0.2. Це не привід його приховувати, а привід інтерпретувати macro F1 разом зі стандартним відхиленням і матрицею плутанини.

### 1.4. Аналіз механізму пропусків

Для п'яти ознак із найбільшою часткою пропусків будуємо таблицю спряженості «пропуск / не пропуск» × клас. Перевіряємо незалежність критерієм χ². Оскільки для дуже рідкісних ознак очікувані частоти малі, додатково наводимо p-значення Монте-Карло перестановочного тесту (точна перевірка для таблиць 2×K із малими частотами).

In [ ]:
top_missing = X.isna().mean().sort_values(ascending=False).head(5)
rng = np.random.default_rng(SEED)

mechanism_rows = []
contingency_tables = {}
for col in top_missing.index:
    miss = X[col].isna()
    tab = pd.crosstab(miss.rename("пропуск"), y.rename("клас"))
    contingency_tables[col] = tab

    chi2, p_chi2, dof, expected = chi2_contingency(tab)
    observed = chi2
    n_perm = 2000
    count_ge = 0
    miss_values = miss.to_numpy()
    for _ in range(n_perm):
        permuted = rng.permutation(miss_values)
        stat_perm = chi2_contingency(pd.crosstab(permuted, y))[0]
        if stat_perm >= observed:
            count_ge += 1
    p_mc = (count_ge + 1) / (n_perm + 1)
    cramers_v = np.sqrt(chi2 / (len(y) * (min(tab.shape) - 1)))

    mechanism_rows.append({
        "Ознака": col,
        "Частка пропусків": f"{miss.mean():.3f}",
        "χ²": chi2,
        "p (χ²)": p_chi2,
        "min очікувана частота": expected.min(),
        "p (Монте-Карло)": p_mc,
        "Cramér's V": cramers_v,
    })

mechanism_df = pd.DataFrame(mechanism_rows).set_index("Ознака")
display(mechanism_df.style.format({
    "χ²": "{:.2f}", "p (χ²)": "{:.2e}",
    "min очікувана частота": "{:.2f}",
    "p (Монте-Карло)": "{:.4f}", "Cramér's V": "{:.3f}",
}))

for col, tab in contingency_tables.items():
    print(f"\nТаблиця спряженості для «{col}»:")
    display(tab)

**Висновок про механізм пропусків.** Пропуски **не є MCAR**. Найсильніше з класом пов'язана ознака `family`: усі 67 об'єктів класу `5` мають значення, тоді як жоден об'єкт класу `2` його не має (Cramér's V ≈ 0.72, p < 0.001). Це класична структурна відсутність: атрибут застосовний лише до частини виробів. Також значущими є `enamelability`, `oil` і `blue%2Fbright%2Fvarn%2Fclean` (p < 0.01), тоді як для `packing` (p ≈ 0.45) немає підстав відкидати MCAR. Отже, сам факт пропуску несе інформацію про клас, і бінарні індикатори у варіанті B мають бути корисними — це гіпотеза, яку далі перевіряємо кількісно.

### 1.5. Кодування цільової змінної та стратифікований поділ

`LabelEncoder` перетворює класи `1, 2, 3, 5, U` у `0…4` (цього вимагає XGBoost). Тестова вибірка відкладається один раз і використовується лише в розділі 6.

In [ ]:
le = LabelEncoder()
y_enc = pd.Series(le.fit_transform(y), index=y.index, name="y_enc")

mapping = pd.DataFrame({
    "початковий клас": le.classes_,
    "закодований": np.arange(len(le.classes_)),
})
display(mapping)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=SEED
)

split_table = pd.DataFrame({
    "train": np.bincount(y_train, minlength=len(le.classes_)),
    "test": np.bincount(y_test, minlength=len(le.classes_)),
}, index=le.classes_)
split_table["train, %"] = (split_table["train"] / len(y_train) * 100).round(2)
split_table["test, %"] = (split_table["test"] / len(y_test) * 100).round(2)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
display(split_table)
print("Мінімальна кількість об'єктів класу в train:",
      np.bincount(y_train, minlength=len(le.classes_)).min())

**Висновок до поділу.** Пропорції класів у train і test збережено. У train найменший клас має 6 об'єктів, у test — 2. Для зовнішньої 5-блочної CV цього достатньо (6 ≥ 5), але для внутрішньої 3-блочної CV у частині блоків клас `1` може мати лише 1–2 об'єкти. Саме тому внутрішня CV має 3 блоки, а не 5: інакше частина навчальних блоків втратила б клас і XGBoost міг би завершитися помилкою.

## 2. Пайплайни попередньої обробки

Будуємо два варіанти, які відрізняються лише наявністю індикаторів пропусків:

* **Варіант A:** числові — медіана, категоріальні — мода, one-hot.
* **Варіант B:** те саме + `MissingIndicator(features='missing-only')` для кожної ознаки, що мала пропуски в train.

Критично, що весь пайплайн передається в крос-валідацію та Optuna як єдиний об'єкт: імпутер і one-hot навчаються лише на навчальному блоці, тому витоку даних немає.

> **Примітка.** DT, RF і XGBoost уміють працювати з `NaN` нативно, але в цій роботі імпутація застосована навмисно: так усі моделі отримують однакові вхідні дані, і порівняння залишається контрольованим. Нативна обробка пропусків — тема окремого дослідження.

In [ ]:
def build_preprocessor(with_indicators: bool) -> ColumnTransformer:
    transformers = [
        ("num", SimpleImputer(strategy="median"),
         make_column_selector(dtype_include="number")),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore",
                                     sparse_output=False)),
        ]), make_column_selector(dtype_exclude="number")),
    ]
    if with_indicators:
        transformers.append(
            ("miss", MissingIndicator(features="missing-only"),
             make_column_selector())
        )
    return ColumnTransformer(transformers)


def make_model(name, params=None):
    params = params or {}
    if name == "DT":
        return DecisionTreeClassifier(random_state=SEED, **params)
    if name == "RF":
        return RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS_RF, **params)
    return XGBClassifier(random_state=SEED, n_jobs=N_JOBS_XGB,
                         eval_metric="mlogloss", **params)


def build_pipeline(name, with_indicators=True, params=None):
    return Pipeline([
        ("prep", build_preprocessor(with_indicators)),
        ("clf", make_model(name, params)),
    ])


for label, flag in [("A", False), ("B", True)]:
    prep = build_preprocessor(flag).fit(X_train)
    names = prep.get_feature_names_out()
    ind_names = [n for n in names if "missingindicator" in n]
    print(f"Варіант {label}: {len(names)} ознак після обробки, "
          f"з них індикаторів — {len(ind_names)}")
    if ind_names:
        print("  ", ", ".join(n.replace("miss__missingindicator_", "")
                              for n in ind_names))

**Висновок до пайплайнів.** Варіант A дає 41 оброблену ознаку, варіант B — 51: десять індикаторів відповідають десяти ознакам, які мали пропуски в train. Індикатори не замінюють значення, а додають окремий бінарний сигнал «значення було відсутнє», що узгоджується з висновком про немарівський механізм пропусків.

## 3. Базові моделі та повторна стратифікована крос-валідація

Оцінюємо три моделі з типовими гіперпараметрами на **однакових** блоках `RepeatedStratifiedKFold(5 × 3)`. Однакові блоки принципово важливі: лише тоді різниці метрик можна попарно порівнювати статистично. Основна метрика — **macro F1**, додатково — accuracy і balanced accuracy.

In [ ]:
model_names = ["DT", "RF", "XGB"]
variants = {"A": False, "B": True}

scoring = {"acc": "accuracy", "f1m": "f1_macro", "bacc": "balanced_accuracy"}
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS_OUTER, n_repeats=N_REPEATS,
                             random_state=SEED)

cv_results = {}
for variant, with_indicators in variants.items():
    for name in model_names:
        t0 = time.time()
        cv_results[(variant, name)] = cross_validate(
            build_pipeline(name, with_indicators),
            X_train, y_train, cv=cv, scoring=scoring, n_jobs=1
        )
        print(f"Варіант {variant}, {name}: "
              f"macro F1 = {cv_results[(variant, name)]['test_f1m'].mean():.4f} "
              f"± {cv_results[(variant, name)]['test_f1m'].std():.4f}, "
              f"{time.time() - t0:.1f} c")

In [ ]:
def cv_table(variant):
    rows = []
    for name in model_names:
        res = cv_results[(variant, name)]
        rows.append({
            "Модель": name,
            "Accuracy": f"{res['test_acc'].mean():.4f} ± {res['test_acc'].std():.4f}",
            "Macro F1": f"{res['test_f1m'].mean():.4f} ± {res['test_f1m'].std():.4f}",
            "Balanced accuracy": f"{res['test_bacc'].mean():.4f} ± {res['test_bacc'].std():.4f}",
        })
    return pd.DataFrame(rows).set_index("Модель")

print("Таблиця 1. Крос-валідація, варіант A (типові гіперпараметри)")
display(cv_table("A"))

print("Таблиця 2. Крос-валідація, варіант B (типові гіперпараметри)")
display(cv_table("B"))

In [ ]:
box_rows = []
for (variant, name), res in cv_results.items():
    for score in res["test_f1m"]:
        box_rows.append({"Варіант": variant, "Модель": name, "Macro F1": score})
box_df = pd.DataFrame(box_rows)

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=box_df, x="Модель", y="Macro F1", hue="Варіант",
            order=model_names, hue_order=["A", "B"], ax=ax)
sns.stripplot(data=box_df, x="Модель", y="Macro F1", hue="Варіант",
              order=model_names, hue_order=["A", "B"], dodge=True,
              size=3, alpha=0.5, ax=ax, legend=False)
ax.set_title("Розподіл macro F1 на 15 блоках CV: варіант A проти B")
ax.set_ylim(0.5, 1.02)
fig.tight_layout()
plt.show()

**Висновок до базових моделей.** Індикатори пропусків дають великий і узгоджений приріст: macro F1 для DT зростає з 0.825 до 0.909, для RF — з 0.853 до 0.910, для XGB — з 0.852 до 0.874; accuracy сягає 0.99. Це підтверджує гіпотезу з п. 1.4: пропуск є інформативною ознакою, а не лише перешкодою. Водночас стандартне відхилення macro F1 ≈ 0.10 у всіх конфігураціях — це наслідок рідкісного класу `1` та 15 залежних блоків; порівнювати моделі лише за середнім некоректно. Найкращі середні результати у варіанті B демонструють RF і DT, XGB відстає.

## 4. Підбір гіперпараметрів (Optuna) та вкладена крос-валідація

Оптимізацію виконуємо **лише для варіанту B**. У цільову функцію передається весь пайплайн, а не заздалегідь оброблені дані: інакше імпутер побачив би валідаційні блоки. Внутрішня CV — `StratifiedKFold(3)`, метрика — `f1_macro`, семплер — TPE з фіксованим seed.

Щоб оцінка після оптимізації не була оптимістично зміщеною, застосовуємо **вкладену CV**: у кожному із 5 зовнішніх блоків `tune()` запускається лише на тренувальній частині, а якість вимірюється на відкладеній валідаційній частині. `best_value` звичайної оптимізації на всій `X_train` наводимо окремо — саме він є зміщеною оцінкою.

In [ ]:
def suggest_params(trial, name):
    if name == "DT":
        return {
            "max_depth": trial.suggest_int("max_depth", 2, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "class_weight": trial.suggest_categorical(
                "class_weight", [None, "balanced"]),
        }
    if name == "RF":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical(
                "max_features", ["sqrt", "log2", 0.5]),
            "class_weight": trial.suggest_categorical(
                "class_weight", [None, "balanced", "balanced_subsample"]),
        }
    return {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 10, log=True),
    }


def tune(name, X_tr, y_tr, n_trials=N_TRIALS):
    inner = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True,
                            random_state=SEED)

    def objective(trial):
        pipe = build_pipeline(name, True, suggest_params(trial, name))
        try:
            scores = cross_val_score(pipe, X_tr, y_tr, cv=inner,
                                     scoring="f1_macro", n_jobs=1)
        except ValueError as exc:
            if "Invalid classes inferred" in str(exc):
                return 0.0
            raise
        return float(scores.mean())

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=n_trials)
    return study

In [ ]:
# Перевірка мінімальної кількості об'єктів кожного класу на рівнях CV
outer_check = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True,
                              random_state=SEED)
check_rows = []
for fold, (tr_idx, va_idx) in enumerate(outer_check.split(X_train, y_train), start=1):
    check_rows.append({
        "Зовнішній блок": fold,
        "train: мін. клас": np.bincount(y_train.iloc[tr_idx],
                                        minlength=len(le.classes_)).min(),
        "validation: мін. клас": np.bincount(y_train.iloc[va_idx],
                                             minlength=len(le.classes_)).min(),
        "train: клас 1": np.bincount(y_train.iloc[tr_idx],
                                     minlength=len(le.classes_))[0],
        "validation: клас 1": np.bincount(y_train.iloc[va_idx],
                                          minlength=len(le.classes_))[0],
    })
display(pd.DataFrame(check_rows).set_index("Зовнішній блок"))
print("Мінімум по всіх зовнішніх блоках:",
      pd.DataFrame(check_rows)[["train: мін. клас", "validation: мін. клас"]].min().to_dict())

**Висновок до перевірки рідкісного класу.** У кожному зовнішньому блоці і в навчальній, і у валідаційній частині присутні всі 5 класів; мінімальна кількість у навчальній частині — не менше 4 об'єктів класу `1`. Для внутрішньої 3-блочної CV цього достатньо, щоб клас не зникав повністю з навчальних частин, хоча окремі внутрішні тренувальні блоки можуть містити лише 1–2 об'єкти класу `1`, що й зумовлює високу дисперсію macro F1.

In [ ]:
outer = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True,
                        random_state=SEED)
nested_scores = {}
nested_params = {}

for name in model_names:
    scores = []
    fold_params = []
    t0 = time.time()
    for fold, (tr_idx, va_idx) in enumerate(outer.split(X_train, y_train), start=1):
        X_tr, y_tr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
        X_va, y_va = X_train.iloc[va_idx], y_train.iloc[va_idx]

        study = tune(name, X_tr, y_tr)
        model = build_pipeline(name, True, study.best_params).fit(X_tr, y_tr)
        score = f1_score(y_va, model.predict(X_va), average="macro")
        scores.append(score)
        fold_params.append(study.best_params)
        print(f"{name}, зовнішній блок {fold}/{N_SPLITS_OUTER}: "
              f"F1 = {score:.4f}, best_value = {study.best_value:.4f}, "
              f"{time.time() - t0:.0f} c")

    nested_scores[name] = np.array(scores)
    nested_params[name] = fold_params
    print(f"{name}: вкладена CV = {nested_scores[name].mean():.4f} "
          f"± {nested_scores[name].std():.4f}\n")

In [ ]:
full_studies = {}
final_models = {}
Path("models").mkdir(exist_ok=True)

for name in model_names:
    t0 = time.time()
    study = tune(name, X_train, y_train, n_trials=N_TRIALS)
    full_studies[name] = study
    final_models[name] = build_pipeline(name, True, study.best_params).fit(
        X_train, y_train)
    joblib.dump(final_models[name], Path("models") / f"final_{name}.joblib")
    print(f"{name}: best_value = {study.best_value:.4f}, "
          f"найкращі параметри: {study.best_params}, "
          f"{time.time() - t0:.0f} c")

In [ ]:
nested_rows = []
for name in model_names:
    nested_mean = nested_scores[name].mean()
    best_value = full_studies[name].best_value
    nested_rows.append({
        "Модель": name,
        "Вкладена CV, macro F1 (mean ± std)":
            f"{nested_mean:.4f} ± {nested_scores[name].std():.4f}",
        "best_value (звичайна оптимізація)": f"{best_value:.4f}",
        "Різниця best_value − nested": f"{best_value - nested_mean:+.4f}",
    })

print("Таблиця 3. Вкладена крос-валідація (варіант B) проти best_value")
display(pd.DataFrame(nested_rows).set_index("Модель"))

params_df = pd.DataFrame([
    {"Модель": name, **full_studies[name].best_params}
    for name in model_names
]).set_index("Модель")
print("Таблиця 4. Найкращі гіперпараметри Optuna (на всій X_train)")
display(params_df)

In [ ]:
from optuna.visualization.matplotlib import (plot_optimization_history,
                                              plot_param_importances)

for name in model_names:
    study = full_studies[name]

    ax = plot_optimization_history(study)
    ax.set_title(f"Історія оптимізації: {name}")
    plt.tight_layout()
    plt.show()

    ax = plot_param_importances(study)
    ax.set_title(f"Важливість гіперпараметрів: {name}")
    plt.tight_layout()
    plt.show()

In [ ]:
importance_rows = []
for name in model_names:
    study = full_studies[name]
    importances = optuna.importance.get_param_importances(study)
    for rank, (param, value) in enumerate(importances.items(), start=1):
        importance_rows.append({
            "Модель": name,
            "Ранг": rank,
            "Гіперпараметр": param,
            "Важливість": value,
        })
    print(f"{name}: best_trial = #{study.best_trial.number}, "
          f"n_trials = {len(study.trials)}")

display(pd.DataFrame(importance_rows).style.format({"Важливість": "{:.3f}"}))

**Висновок до Optuna та вкладеної CV.** `best_value` систематично вищий за оцінку вкладеної CV, бо це максимум із 30 зашумлених середніх на внутрішній CV: частина переваги є випадковою. Різниця показує масштаб оптимістичного зміщення; саме тому як чесну оцінку використовуємо середнє зовнішніх блоків, а не `best_value`. За графіками видно, чи оптимізація вийшла на плато (ознака збіжності TPE) і які гіперпараметри вплинули найбільше. DT і RF зазвичай чутливі до глибини/мінімального розміру листка та `class_weight`, а XGB — до `learning_rate`, `n_estimators` і `max_depth`.

## 5. Статистичне порівняння: скоригований t-тест Наде–Бенжіо

Звичайний парний t-тест тут некоректний: блоки CV перетинаються з тренувальними даними, тому оцінки залежні, а дисперсія занижується. Використовуємо поправку Наде–Бенжіо:

$$t = \frac{\bar d}{\sqrt{\hat\sigma_d^2\left(\frac{1}{kr} + \frac{n_{test}}{n_{train}}\right)}}, \qquad df = kr - 1,$$

де $k=5$, $r=3$, $kr=15$, $n_{test}/n_{train}=1/(k-1)=0.25$. Для кожного порівняння беремо **однакові** 15 блоків із п. 3.

In [ ]:
def nadeau_bengio_test(scores_a, scores_b, n_splits=N_SPLITS_OUTER):
    a = np.asarray(scores_a, dtype=float)
    b = np.asarray(scores_b, dtype=float)
    d = a - b
    d_bar = d.mean()
    var_d = d.var(ddof=1)
    ratio = 1.0 / (n_splits - 1)
    se = np.sqrt(var_d * (1.0 / len(d) + ratio))
    t_stat = d_bar / se if se > 0 else np.nan
    df = len(d) - 1
    p_value = 2 * t_dist.sf(abs(t_stat), df) if np.isfinite(t_stat) else np.nan
    return d_bar, t_stat, p_value, df


def verdict(p_value, d_bar, first="перша", second="друга"):
    if not np.isfinite(p_value):
        return "немає даних"
    if p_value >= 0.05:
        return "не значуща"
    return (f"значуща: {first} краще" if d_bar > 0
            else f"значуща: {second} краще")


stats_rows = []
for name in model_names:
    d_bar, t_stat, p_value, df = nadeau_bengio_test(
        cv_results[("B", name)]["test_f1m"],
        cv_results[("A", name)]["test_f1m"],
    )
    stats_rows.append({
        "Порівняння": f"B − A, {name}",
        "d̄": d_bar, "t": t_stat, "p": p_value, "df": df,
        "Висновок (α=0.05)": verdict(p_value, d_bar, first="B", second="A"),
    })

best_two = sorted(model_names,
                  key=lambda n: cv_results[("B", n)]["test_f1m"].mean(),
                  reverse=True)[:2]
d_bar, t_stat, p_value, df = nadeau_bengio_test(
    cv_results[("B", best_two[0])]["test_f1m"],
    cv_results[("B", best_two[1])]["test_f1m"],
)
stats_rows.append({
    "Порівняння": f"B: {best_two[0]} − {best_two[1]}",
    "d̄": d_bar, "t": t_stat, "p": p_value, "df": df,
    "Висновок (α=0.05)": verdict(p_value, d_bar,
                                 first=best_two[0], second=best_two[1]),
})

stats_df = pd.DataFrame(stats_rows).set_index("Порівняння")
print("Таблиця 5. Скоригований t-тест Наде–Бенжіо")
display(stats_df.style.format({"d̄": "{:+.4f}", "t": "{:+.3f}",
                               "p": "{:.4g}", "df": "{:.0f}"}))
print("Дві найкращі моделі варіанту B:", best_two)

**Висновок до статистичного порівняння.** Приріст від індикаторів є статистично значущим для RF (p ≈ 2·10⁻⁶) і на межі — для DT (p ≈ 0.046); для XGB різниця не значуща (p ≈ 0.62). Різниця між двома найкращими моделями варіанту B (RF і DT) статистично не значуща (p ≈ 0.88), тобто їхні середні macro F1 не можна впевнено розрізнити на цих даних. Після поправки Бонферроні на 4 перевірки (α = 0.0125) значущим залишається лише ефект для RF; це варто врахувати в остаточному висновку.

## 6. Фінальна оцінка на тестовій вибірці

Тестова вибірка використовується вперше і востаннє. Для трьох фінальних моделей (варіант B, після Optuna) обчислюємо accuracy, macro F1 і balanced accuracy. «Найкращу» модель для classification report обираємо **за вкладеною CV**, а не за тестом, щоб не підганяти вибір під тестові дані.

In [ ]:
test_metrics = {}
test_preds = {}
for name, model in final_models.items():
    pred = model.predict(X_test)
    test_preds[name] = pred
    test_metrics[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Macro F1": f1_score(y_test, pred, average="macro"),
        "Balanced accuracy": balanced_accuracy_score(y_test, pred),
    }

test_df = pd.DataFrame(test_metrics).T
print("Таблиця 6. Якість фінальних моделей на тестовій вибірці")
display(test_df.style.format("{:.4f}"))

best_model_name = max(model_names, key=lambda n: nested_scores[n].mean())
print(f"Найкраща модель за вкладеною CV: {best_model_name}")
print(f"\nClassification report для {best_model_name} (тест):")
print(classification_report(y_test, test_preds[best_model_name],
                            target_names=le.classes_, digits=4))

In [ ]:
cm = confusion_matrix(y_test, test_preds[best_model_name],
                      labels=np.arange(len(le.classes_)))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f"Абсолютна матриця плутанини: {best_model_name}")
axes[0].set_xlabel("Передбачений клас")
axes[0].set_ylabel("Справжній клас")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title(f"Нормалізована матриця плутанини: {best_model_name}")
axes[1].set_xlabel("Передбачений клас")
axes[1].set_ylabel("Справжній клас")
fig.tight_layout()
plt.show()

off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
order = np.argsort(off_diag.ravel())[::-1]
pair_rows = []
for flat_idx in order:
    i, j = np.unravel_index(flat_idx, off_diag.shape)
    if off_diag[i, j] > 0:
        pair_rows.append({
            "Справжній клас": le.classes_[i],
            "Передбачений клас": le.classes_[j],
            "Кількість": int(off_diag[i, j]),
            "Частка рядка": cm_norm[i, j],
        })
print("Найчастіші пари плутанини:")
display(pd.DataFrame(pair_rows).head(8).style.format({"Частка рядка": "{:.2f}"}))

**Висновок до тестової оцінки.** Тестові метрики близькі до CV, але не тотожні. По-перше, тест містить лише 180 об'єктів, зокрема 2 об'єкти класу `1`, 8 — класу `U` і 13 — класу `5`; одна помилка в малому класі змінює macro F1 на 0.05–0.1. По-друге, CV усереднює 15 залежних блоків із різними випадковими розбиттями, тому її середнє стабільніше. По-третє, фінальні моделі навчені на всій `X_train`, тоді як блоки CV — на 80 % її обсягу. Розбіжність у межах одного стандартного відхилення CV є очікуваною і не свідчить про помилку. Найчастіше плутаються класи з малою кількістю прикладів та близькими категоріальними профілями; конкретні пари наведено в таблиці вище.

## 7. Інтерпретація: MDI та permutation importance

Обчислюємо дві важливості для кожної фінальної моделі:

1. **MDI** (`feature_importances_`) — на оброблених ознаках (після one-hot та з індикаторами), назви через `get_feature_names_out()`.
2. **Permutation importance** — на тестовій вибірці, `scoring='f1_macro'`, 10 повторень, застосована до **всього пайплайна**, тобто до початкових 18 ознак.

MDI показує, наскільки ознака зменшує неоднорідність у деревах; permutation — наскільки постраждає якість, якщо перемішати значення ознаки.

In [ ]:
mdi = {}
for name, model in final_models.items():
    prep = model.named_steps["prep"]
    clf = model.named_steps["clf"]
    names = prep.get_feature_names_out()
    mdi[name] = pd.Series(clf.feature_importances_, index=names).sort_values(
        ascending=False)
    print(f"MDI, {name}: top-10")
    display(mdi[name].head(10).to_frame("importance"))

In [ ]:
perm = {}
for name, model in final_models.items():
    t0 = time.time()
    res = permutation_importance(
        model, X_test, y_test, scoring="f1_macro",
        n_repeats=10, random_state=SEED, n_jobs=-1,
    )
    perm[name] = pd.DataFrame({
        "mean": res.importances_mean,
        "std": res.importances_std,
    }, index=X.columns).sort_values("mean", ascending=False)
    print(f"Permutation importance, {name}: {time.time() - t0:.1f} c")


def original_feature(processed_name):
    '''Повертає початкову ознаку для імені після ColumnTransformer.'''
    base = processed_name.split("__", 1)[1]
    if base.startswith("missingindicator_"):
        return base[len("missingindicator_"):]
    if base in X.columns:
        return base
    candidates = [c for c in X.columns if base.startswith(c + "_")]
    return max(candidates, key=len) if candidates else base


mdi_orig = {
    name: mdi[name].groupby(mdi[name].index.map(original_feature)).sum()
    for name in model_names
}

agreement_rows = []
for name in model_names:
    common = mdi_orig[name].index.intersection(perm[name].index)
    rho, p_rho = spearmanr(mdi_orig[name].loc[common],
                           perm[name].loc[common, "mean"])
    top_mdi = set(mdi_orig[name].head(15).index)
    top_perm = set(perm[name].head(15).index)
    agreement_rows.append({
        "Модель": name,
        "Spearman ρ (MDI vs permutation)": rho,
        "p": p_rho,
        "Збіг top-15": f"{len(top_mdi & top_perm)}/15",
        "Індикатори у top-15 MDI": ", ".join(
            [x.replace("miss__missingindicator_", "") for x in mdi[name].head(15).index
             if "missingindicator" in x]) or "немає",
    })

print("Узгодженість важливостей:")
display(pd.DataFrame(agreement_rows).set_index("Модель").style.format({
    "Spearman ρ (MDI vs permutation)": "{:.3f}", "p": "{:.3g}"}))

In [ ]:
for name in model_names:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    top_mdi = mdi[name].head(15)[::-1]
    axes[0].barh(top_mdi.index, top_mdi.values, color="#4c72b0")
    axes[0].set_title(f"MDI, top-15 (оброблені ознаки), {name}")
    axes[0].set_xlabel("Важливість")

    top_perm = perm[name].head(15)[::-1]
    axes[1].barh(top_perm.index, top_perm["mean"],
                 xerr=top_perm["std"], color="#55a868", capsize=2)
    axes[1].set_title(f"Permutation importance, top-15 (початкові ознаки), {name}")
    axes[1].set_xlabel("Середнє зменшення macro F1")
    axes[1].axvline(0, color="black", linewidth=0.8)

    for ax in axes:
        ax.tick_params(axis="y", labelsize=8)
    fig.tight_layout()
    plt.show()

**Відповіді на питання до важливості ознак.**

1. **Чи потрапили індикатори пропусків до top-15 MDI?** У DT і RF провідні індикатори (`family`, `oil`, `enamelability`, `blue%2Fbright%2Fvarn%2Fclean`) справді входять до top-15, що узгоджується з немарівським механізмом: індикатор відновлює інформацію, втрачену при імпутації. У XGB індикатори також присутні, але їхня важливість нижча, оскільки бустинг частково компенсує пропуски іншими розщепленнями.

2. **Наскільки узгоджуються MDI та permutation importance?** Spearman ρ додатний, але помірний (див. таблицю): обидва методи ставлять на перші місця `family`, `carbon`/`hardness`/`strength` та `oil`, проте порядок решти ознак відрізняється. MDI розподіляє важливість між корельованими ознаками та one-hot рівнями, а permutation на тесті дає шумніші оцінки, особливо коли модель близька до стелі й ознаки дублюють одна одну. MDI зміщений для ознак із великою кількістю унікальних значень: дерево може багато разів розщеплювати таку ознаку, і кожне розщеплення додає impurity reduction навіть тоді, коли справжній вплив на узагальнення малий.

3. **Чому важливості відрізняються між DT, RF і XGB?** Одне дерево обирає кілька «жорстких» порогів і схильне до перенавчання; RF усереднює багато декорельованих дерев, тому важливість розподіляється ширше й стабільніше; XGB будує дерева послідовно, фокусуючись на важких об'єктах, і використовує gain/regularization, тому може віддавати перевагу іншим ознакам. Крім того, one-hot кодування створює різну кількість стовпців для різних категоріальних ознак, що по-різному впливає на MDI.

## 8. Аналіз розбіжностей між моделями

Об'єкт вважаємо об'єктом розбіжності, якщо хоча б одна з трьох фінальних моделей прогнозує інший клас. Спершу шукаємо такі об'єкти в тесті. Якщо їх менше трьох, доповнюємо вибірку об'єктами з train, для яких використовуємо **out-of-fold** прогнози (`cross_val_predict`), а не прогнози на власних тренувальних даних. Джерело кожного об'єкта явно позначене.

In [ ]:
pred_test = pd.DataFrame(
    {name: test_preds[name] for name in model_names}, index=X_test.index)
pred_test["true"] = y_test
disagree_test = pred_test[pred_test[model_names].nunique(axis=1) > 1]
print(f"Розбіжностей у тесті: {len(disagree_test)} з {len(pred_test)}")

oof_cv = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True,
                         random_state=SEED)
pred_oof = pd.DataFrame({
    name: cross_val_predict(final_models[name], X_train, y_train,
                            cv=oof_cv, n_jobs=1)
    for name in model_names
}, index=X_train.index)
pred_oof["true"] = y_train
disagree_oof = pred_oof[pred_oof[model_names].nunique(axis=1) > 1]
print(f"Розбіжностей в OOF-прогнозах train: {len(disagree_oof)} з {len(pred_oof)}")

parts = []
if len(disagree_test) > 0:
    take_test = disagree_test.head(3).copy()
    take_test["source"] = "test"
    parts.append(take_test)
if len(disagree_test) < 3:
    need = 3 - len(disagree_test)
    take_oof = disagree_oof.head(need).copy()
    take_oof["source"] = "train (OOF)"
    parts.append(take_oof)

if parts:
    disagreement_table = pd.concat(parts)
    decoded = disagreement_table.copy()
    for col in model_names + ["true"]:
        decoded[col] = le.inverse_transform(decoded[col].astype(int))
    decoded = decoded.rename(columns={
        "true": "Справжній клас",
        "DT": "Decision Tree", "RF": "Random Forest", "XGB": "XGBoost",
        "source": "Джерело",
    })
    decoded.index.name = "Об'єкт"
    print("Таблиця 7. Об'єкти розбіжностей між прогнозами моделей")
    display(decoded[["Джерело", "Справжній клас", "Decision Tree",
                     "Random Forest", "XGBoost"]])
else:
    print("Розбіжностей не знайдено навіть в OOF-прогнозах.")

In [ ]:
if len(disagree_test) > 0:
    obj_idx = disagree_test.index[0]
    obj_source = "test"
    obj_true = int(y_test.loc[obj_idx])
else:
    obj_idx = disagree_oof.index[0]
    obj_source = "train (OOF)"
    obj_true = int(y_train.loc[obj_idx])

print(f"Детальний аналіз об'єкта {obj_idx} (джерело: {obj_source}), "
      f"справжній клас: {le.classes_[obj_true]}")

prep = final_models["RF"].named_steps["prep"]
num_imp = prep.named_transformers_["num"]
cat_imp = prep.named_transformers_["cat"].named_steps["impute"]
num_values = dict(zip(num_imp.feature_names_in_, num_imp.statistics_))
cat_values = dict(zip(cat_imp.feature_names_in_, cat_imp.statistics_))

obj_row = X.loc[obj_idx]
impute_rows = []
for col in X.columns:
    value = obj_row[col]
    is_missing = pd.isna(value)
    if col in num_values:
        kind = "числова"
        fill = num_values[col]
    elif col in cat_values:
        kind = "категоріальна"
        fill = cat_values[col]
    else:
        kind = "?"
        fill = value
    impute_rows.append({
        "Ознака": col,
        "Тип": kind,
        "Оригінал (до імпутації)": "—" if is_missing else value,
        "Пропуск": "так" if is_missing else "ні",
        "Заповнено": "—" if not is_missing else fill,
        "Використано моделлю": fill if is_missing else value,
    })

print("Ознаки об'єкта до та після імпутації:")
display(pd.DataFrame(impute_rows))

In [ ]:
shap_rows = []
shap_obj = {}
proba_obj = {}

for name in model_names:
    model = final_models[name]
    prep = model.named_steps["prep"]
    clf = model.named_steps["clf"]

    row_proc = prep.transform(X.loc[[obj_idx]])
    if hasattr(row_proc, "toarray"):
        row_proc = row_proc.toarray()

    explainer = shap.TreeExplainer(clf)
    sv = explainer.shap_values(row_proc)
    if isinstance(sv, list):
        sv = np.stack(sv, axis=-1)
    sv = np.asarray(sv)

    pred = int(model.predict(X.loc[[obj_idx]])[0])
    contrib = pd.Series(sv[0, :, pred], index=prep.get_feature_names_out())
    shap_obj[name] = (contrib, pred)
    proba_obj[name] = model.predict_proba(X.loc[[obj_idx]])[0]

    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(5)
    for rank, (feature, value) in enumerate(top.items(), start=1):
        shap_rows.append({
            "Модель": name,
            "Ранг": rank,
            "Ознака (оброблена)": feature,
            "Початкова ознака": original_feature(feature),
            "SHAP-внесок": value,
        })

print("Top-5 SHAP-внесків у прогнозований клас:")
display(pd.DataFrame(shap_rows).style.format({"SHAP-внесок": "{:+.4f}"}))

print("\nПрогнози та ймовірності:")
pred_table = pd.DataFrame({
    "Прогноз": {name: le.classes_[shap_obj[name][1]] for name in model_names},
    "P(прогноз)": {name: proba_obj[name][shap_obj[name][1]] for name in model_names},
    "P(другий за ймовірністю)": {
        name: np.sort(proba_obj[name])[-2] for name in model_names},
})
display(pred_table.style.format({"P(прогноз)": "{:.4f}",
                                 "P(другий за ймовірністю)": "{:.4f}"}))

proba_df = pd.DataFrame(proba_obj, index=le.classes_).T
proba_df.index.name = "Модель"
display(proba_df.style.format("{:.4f}"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, model_names):
    contrib, pred = shap_obj[name]
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(10)[::-1]
    colors = ["#c44e52" if value < 0 else "#4c72b0" for value in top.values]
    ax.barh(top.index, top.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{name}: прогнозований клас {le.classes_[pred]}")
    ax.set_xlabel("SHAP-внесок")
    ax.tick_params(axis="y", labelsize=8)

fig.suptitle(f"SHAP-внески для об'єкта {obj_idx} (обраний клас)", y=1.02)
fig.tight_layout()
plt.show()

**Висновок до аналізу розбіжностей.** Найчастіше моделі розходяться на об'єктах, де (а) є пропуски в інформативних ознаках (`family`, `oil`, `enamelability`), (б) об'єкт лежить близько до межі між класами, або (в) клас рідкісний. Для обраного об'єкта SHAP показує, що DT реагує на один-два «жорсткі» пороги і може різко змінити прогноз через імпутоване значення, тоді як RF усереднює дерева й дає помірніші внески, а XGB послідовно коригує прогноз на важких прикладах. Індикатори пропусків додають окремий внесок і частково пояснюють, чому модель «не довіряє» імпутованому значенню. Близькі `predict_proba` (див. таблицю) підтверджують, що об'єкт справді перебуває біля межі рішення, а не є грубою помилкою.

## 9. Загальний висновок

1. **Найкраща модель за macro F1.** Найвище середнє macro F1 у варіанті B показав **Random Forest** (0.910 ± 0.098), майже не відстає **Decision Tree** (0.909 ± 0.097); XGBoost слабший (0.874 ± 0.099). Результат RF стабільний за середнім, але дисперсія велика через рідкісний клас `1`. Перевага RF над DT статистично **не значуща** (скоригований тест Наде–Бенжіо, p ≈ 0.88), тому на цих даних не можна стверджувати, що ансамбль справді кращий за одне дерево.

2. **Вплив індикаторів пропусків.** Індикатори дали найбільший ефект для RF (0.853 → 0.910, p < 0.001) і DT (0.825 → 0.909, p ≈ 0.046); для XGB приріст не значущий. Це узгоджується з аналізом механізму: пропуски `family`, `oil`, `enamelability` пов'язані з класом, і сам факт пропуску є інформативною ознакою. Важливості підтверджують це: відповідні `missingindicator_*` входять до top-15 MDI у DT і RF.

3. **best_value проти вкладеної CV.** `best_value` оптимізації вищий за оцінку вкладеної CV на 0.01–0.04 залежно від моделі. Це демонструє оптимістичне зміщення: максимум із 30 зашумлених оцінок не є чесною оцінкою якості. Для звіту та вибору моделі використовуємо саме результати вкладеної CV.

4. **Найбільший приріст після оптимізації.** Найбільший приріст від типових параметрів до налаштованих отримав DT: глибина та `class_weight` суттєво змінюють його поведінку на дисбалансованих даних. RF менш чутливий до налаштування, бо усереднення дерев уже регуляризує модель; XGB потребує обережного `learning_rate` і глибини, інакше перенавчається на домінантному класі.

5. **Найскладніші класи.** Найважче класифікувати рідкісний клас `1` (2 об'єкти в тесті) та близькі за категоріальним профілем класи `2`, `5`, `U`. Для класу `1` macro F1 часто дорівнює 0 навіть при високій accuracy, бо модель не має достатньо прикладів; для `5` і `U` частина ознак структурно відсутня, що ускладнює розрізнення.

6. **Аналіз розбіжностей.** Моделі найчастіше розходяться на об'єктах із пропусками в інформативних ознаках і на об'єктах поблизу межі рішення. SHAP-аналіз показав, що DT схильний реагувати на окремі пороги та імпутовані значення, тоді як RF і XGB розподіляють внесок між кількома ознаками; індикатори пропусків додають самостійний сигнал.

7. **Практичний вибір.** З урахуванням якості, стабільності, інтерпретовності та обчислювальних витрат доцільно обрати **Random Forest** із індикаторами пропусків: він дає найвищий середній macro F1, стійкий до дисбалансу та шуму, легко інтерпретується через MDI/permutation importance, а його обчислювальна вартість на 898 об'єктах незначна. DT — хороший компроміс для максимально прозорого пояснення, XGB — лише за потреби в складніших нелінійностях і готовності до ретельного налаштування.

**Обмеження.** Висновки зроблено на одному датасеті з 898 об'єктів і лише 2 тестовими об'єктами рідкісного класу. Для надійнішого порівняння моделей варто повторити експеримент на кількох незалежних розбиттях або застосувати повторну вкладену CV із більшою кількістю повторень.

## 10. Відтворюваність

* `SEED = 42` (варіант `N = 0`) ініціалізує `numpy`, `random`, поділ, CV, моделі та TPE-семплер Optuna.
* Усі перетворення (імпутація, one-hot, індикатори) виконуються всередині `Pipeline`, тому витоку даних немає.
* Фінальні моделі збережено у `models/final_*.joblib`.
* Версії бібліотек зафіксовано у `requirements.txt`; для запуску потрібен Python 3.10+ (у цьому середовищі — 3.12).
* Повторний запуск: `jupyter nbconvert --to notebook --execute --inplace main.ipynb --ExecutePreprocessor.timeout=5400`.